# Day 04 下午：电商用户行为数据清洗项目

**项目数据：** E Commerce Dataset.xlsx（E Comm 工作表）  
**项目目标：** 将上午学习的处理方法固化为可复用的数据清洗流程，并交付可供第五天分析使用的数据文件。

## 最终交付物

运行本 Notebook 后，应在 output/day04_project/ 中生成：

1. ecommerce_customer_cleaned.csv：清洗后的用户数据；
2. data_quality_before.csv：清洗前质量报告；
3. data_quality_after.csv：清洗后质量报告；
4. cleaning_log.csv：数据处理日志。

## 项目规则

- 原始数据只读，不覆盖；
- 清洗函数接收 DataFrame，返回清洗结果与处理日志；
- 处理规则必须可解释；
- 不使用 Churn 分组填补特征，避免将目标变量信息带入特征处理；
- 发现候选异常值后，先记录和判断，不盲目删除。

---
## 1. 项目初始化与数据读取

In [1]:
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.2f}")

candidates = [
    Path(r"E:\pythonProject\day4\data\E Commerce Dataset.xlsx"),
    Path("data/E Commerce Dataset.xlsx"),
    Path("../data/E Commerce Dataset.xlsx"),
    Path("/Users/yq/muc_training/data/E Commerce Dataset.xlsx"),
]
DATA_PATH = next((path for path in candidates if path.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError("未找到 E Commerce Dataset.xlsx，请修改 DATA_PATH。")

root_candidates = [Path.cwd(), Path.cwd().parent, Path("/Users/yq/Desktop/muc")]
PROJECT_ROOT = next(
    (path for path in root_candidates if (path / "notebooks").exists()),
    Path.cwd()
)
OUTPUT_DIR = PROJECT_ROOT / "output" / "day04_project"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

raw_df = pd.read_excel(DATA_PATH, sheet_name="E Comm")

print(f"原始数据：{DATA_PATH}")
print(f"项目输出目录：{OUTPUT_DIR}")
print(f"原始数据形状：{raw_df.shape}")
raw_df.head()

原始数据：E:\pythonProject\day4\data\E Commerce Dataset.xlsx
项目输出目录：e:\pythonProject\day4\output\day04_project
原始数据形状：(5630, 20)


,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount
0,50001,1,4.00,Mobile Phone,3,6.00,Debit Card,Female,3.00,3,Laptop & Accessory,2,Single,9,1,11.00,1.00,1.00,5.00,159.93
1,50002,1,NaN,Phone,1,8.00,UPI,Male,3.00,4,Mobile,3,Single,7,1,15.00,0.00,1.00,0.00,120.90
2,50003,1,NaN,Phone,1,30.00,Debit Card,Male,2.00,4,Mobile,3,Single,6,1,14.00,0.00,1.00,3.00,120.28
3,50004,1,0.00,Phone,3,15.00,Debit Card,Male,2.00,4,Laptop & Accessory,5,Single,8,0,23.00,0.00,1.00,3.00,134.07
4,50005,1,0.00,Phone,1,12.00,CC,Male,NaN,3,Mobile,5,Single,3,0,11.00,1.00,1.00,3.00,129.60


### 任务 1：确认项目对象

请回答：

1. 每条记录代表什么？
2. 项目的目标变量是哪一列？
3. 为什么 CustomerID 不应作为普通连续数值参与后续分析？

In [ ]:
# 在此写下你的答案：
# 1. 每条记录代表一位电商平台的用户。数据集中每一行对应一个独立客户，
#    记录了该客户的基本信息、使用行为、消费习惯以及是否流失（Churn）等指标。
#
# 2. 项目的目标变量是 Churn（客户流失）。从项目规则中"不使用 Churn
#    分组填补特征，避免将目标变量信息带入特征处理"可以明确确认。
#
# 3. CustomerID 是客户的唯一标识符（名义变量/身份标签），不具备数值大小
#    的实际意义——CustomerID=100 的客户并不比 CustomerID=50 的客户"大两倍"。
#    如果把它当作连续数值特征输入模型，模型会试图从 ID 的数值大小中"学习规律"，
#    导致严重的过拟合（记住特定 ID 而不是学到通用的行为模式），
#    在预测新客户时完全失效。正确的做法是把它作为索引或直接删除。

---
## 2. 构建数据质量报告

质量报告至少应包含字段类型、缺失数量、缺失比例和唯一值数量。它用于对比清洗前后数据质量。

In [2]:
def build_quality_report(data):
    """返回字段级数据质量报告。"""
    report = pd.DataFrame({
        "字段名": data.columns,
        "数据类型": [data[col].dtype for col in data.columns],
        "缺失数量": [data[col].isna().sum() for col in data.columns],
        "缺失比例(%)": [(data[col].isna().sum() / len(data) * 100) for col in data.columns],
        "唯一值数量": [data[col].nunique() for col in data.columns],
    })
    report.set_index("字段名", inplace=True)
    return report

# 生成清洗前质量报告
quality_before = build_quality_report(raw_df)
display(quality_before)

,数据类型,缺失数量,缺失比例(%),唯一值数量
字段名,,,,
CustomerID,int64,0,0.00,5630
Churn,int64,0,0.00,2
Tenure,float64,264,4.69,36
PreferredLoginDevice,str,0,0.00,3
CityTier,int64,0,0.00,3
WarehouseToHome,float64,251,4.46,34
PreferredPaymentMode,str,0,0.00,7
Gender,str,0,0.00,2
HourSpendOnApp,float64,255,4.53,6


### 任务 2：完成初始审计

除字段级质量报告外，请输出：

- 原始数据的完全重复行数；
- CustomerID 重复数量；
- Churn 的频数和流失率；
- 主要类别字段的频数。

In [3]:
# 完成项目初始审计
print("完全重复行数：", raw_df.duplicated().sum())
print("CustomerID 重复数量：", raw_df["CustomerID"].duplicated().sum())
print(raw_df["Churn"].value_counts())
print("流失率：", f"{raw_df['Churn'].mean():.2%}")

for col in ["PreferredLoginDevice", "PreferredPaymentMode", "PreferedOrderCat"]:
    print(f"\n{col}")
    print(raw_df[col].value_counts())

完全重复行数： 0
CustomerID 重复数量： 0
Churn
0    4682
1     948
Name: count, dtype: int64
流失率： 16.84%

PreferredLoginDevice
PreferredLoginDevice
Mobile Phone    2765
Computer        1634
Phone           1231
Name: count, dtype: int64

PreferredPaymentMode
PreferredPaymentMode
Debit Card          2314
Credit Card         1501
E wallet             614
UPI                  414
COD                  365
CC                   273
Cash on Delivery     149
Name: count, dtype: int64

PreferedOrderCat
PreferedOrderCat
Laptop & Accessory    2050
Mobile Phone          1271
Fashion                826
Mobile                 809
Grocery                410
Others                 264
Name: count, dtype: int64


---
## 3. 定义清洗规则

本项目采用以下规则：

| 问题 | 处理规则 | 理由 |
|---|---|---|
| 数值字段缺失 | 使用总体中位数填补 | 稳健且不将缺失误解为 0 |
| Phone / Mobile Phone | 统一为 Mobile Phone | 同一业务类别 |
| COD / Cash on Delivery | 统一为 Cash on Delivery | 同一业务类别 |
| CC / Credit Card | 统一为 Credit Card | 同一业务类别 |
| Mobile / Mobile Phone | 统一为 Mobile Phone | 同一业务类别 |
| 完全重复行 | 若存在则删除 | 完全相同的记录不增加信息 |
| 业务不合规值 | 记录并复核 | 本数据不应仅凭 IQR 直接删除 |

注意：不按 Churn 分组填补缺失值。

In [5]:
NUMERIC_MISSING_COLS = [
    "Tenure",
    "WarehouseToHome",
    "HourSpendOnApp",
    "OrderAmountHikeFromlastYear",
    "CouponUsed",
    "OrderCount",
    "DaySinceLastOrder",
]

CATEGORY_MAPPINGS = {
    "PreferredLoginDevice": {
        "Phone": "Mobile Phone"
    },
    "PreferredPaymentMode": {
        "COD": "Cash on Delivery",
        "CC": "Credit Card"
    },
    "PreferedOrderCat": {
        "Mobile": "Mobile Phone"
    }
}

---
## 4. 编写可复用清洗函数

函数要求：

- 不直接修改传入的原始 DataFrame；
- 返回 cleaned_df 和 cleaning_log；
- 日志至少包含处理步骤、处理规则、处理前记录数、处理后记录数、影响记录数；
- 完成重复值处理、缺失值处理、类别标准化和必要的数据类型转换。

In [6]:
def clean_ecommerce_data(data):
    """
    清洗电商用户行为数据。

    参数：
        data: 原始用户行为 DataFrame

    返回：
        cleaned_df: 清洗后的 DataFrame
        cleaning_log: 处理日志 DataFrame
    """
    # 复制数据，避免覆盖原始数据
    df = data.copy()
    n_before = len(df)

    # 创建日志列表
    logs = []

    # --- 1. 删除完全重复行 --- dup_count = df.duplicated().sum()
    df = df.drop_duplicates()
    n_after_dedup = len(df)
    logs.append({
        "处理步骤": "删除完全重复行",
        "处理规则": "删除所有列完全相同的记录",
        "处理前记录数": n_before,
        "处理后记录数": n_after_dedup,
        "影响记录数": n_before - n_after_dedup,
    })

    # --- 2. 数值字段缺失值填补（使用总体中位数） ---
    for col in NUMERIC_MISSING_COLS:
        missing_count = df[col].isna().sum()
        if missing_count > 0:
            median_val = df[col].median()
            df[col] = df[col].fillna(median_val)
            logs.append({
                "处理步骤": "缺失值填补",
                "处理规则": f"{col} 使用总体中位数 {median_val:.2f} 填补",
                  "处理前记录数": n_after_dedup,
                "处理后记录数": n_after_dedup,
                "影响记录数": missing_count,
            })

    # --- 3. 类别标准化 ---
    for col, mapping in CATEGORY_MAPPINGS.items():
        for old_val, new_val in mapping.items():
            affected = (df[col] == old_val).sum()
            if affected > 0:
                df[col] = df[col].replace({old_val: new_val})
                logs.append({
                    "处理步骤": "类别标准化",
                    "处理规则": f"{col}: '{old_val}' → '{new_val}'",
                    "处理前记录数": n_after_dedup,
                    "处理后记录数": n_after_dedup,
                    "影响记录数": affected,
                })
    # --- 4. Churn 和 Complain 转为整数类型 ---
    for col in ["Churn", "Complain"]:
        if col in df.columns:
            df[col] = df[col].astype(int)

    # 构建日志 DataFrame
    cleaning_log = pd.DataFrame(logs)
    return df, cleaning_log

### 任务 3：运行清洗函数并查看日志

In [7]:
cleaned_df, cleaning_log = clean_ecommerce_data(raw_df)

display(cleaning_log)
cleaned_df.head()

,处理步骤,处理规则,处理前记录数,处理后记录数,影响记录数
0,删除完全重复行,删除所有列完全相同的记录,5630,5630,0
1,缺失值填补,Tenure 使用总体中位数 9.00 填补,5630,5630,264
2,缺失值填补,WarehouseToHome 使用总体中位数 14.00 填补,5630,5630,251
3,缺失值填补,HourSpendOnApp 使用总体中位数 3.00 填补,5630,5630,255
4,缺失值填补,OrderAmountHikeFromlastYear 使用总体中位数 15.00 填补,5630,5630,265
5,缺失值填补,CouponUsed 使用总体中位数 1.00 填补,5630,5630,256
6,缺失值填补,OrderCount 使用总体中位数 2.00 填补,5630,5630,258
7,缺失值填补,DaySinceLastOrder 使用总体中位数 3.00 填补,5630,5630,307
8,类别标准化,PreferredLoginDevice: 'Phone' → 'Mobile Phone',5630,5630,1231
9,类别标准化,PreferredPaymentMode: 'COD' → 'Cash on Delivery',5630,5630,365


,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount
0,50001,1,4.00,Mobile Phone,3,6.00,Debit Card,Female,3.00,3,Laptop & Accessory,2,Single,9,1,11.00,1.00,1.00,5.00,159.93
1,50002,1,9.00,Mobile Phone,1,8.00,UPI,Male,3.00,4,Mobile Phone,3,Single,7,1,15.00,0.00,1.00,0.00,120.90
2,50003,1,9.00,Mobile Phone,1,30.00,Debit Card,Male,2.00,4,Mobile Phone,3,Single,6,1,14.00,0.00,1.00,3.00,120.28
3,50004,1,0.00,Mobile Phone,3,15.00,Debit Card,Male,2.00,4,Laptop & Accessory,5,Single,8,0,23.00,0.00,1.00,3.00,134.07
4,50005,1,0.00,Mobile Phone,1,12.00,Credit Card,Male,3.00,3,Mobile Phone,5,Single,3,0,11.00,1.00,1.00,3.00,129.60


---
## 5. 数据转换与候选异常值检查

为便于第五天分析，请新增：

- TenureGroup：用户使用时长分层；
- IsMobileLogin：是否主要使用移动端登录；
- 候选异常值报告：WarehouseToHome、OrderCount、CashbackAmount。

候选异常值只记录，不在本项目中自动删除。

In [8]:
def iqr_outlier_summary(series):
    """输出 IQR 候选异常值摘要。"""
    series = series.dropna()
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    return {
        "Q1": q1,
        "Q3": q3,
        "下限": lower,
        "上限": upper,
        "候选异常值数量": int(((series < lower) | (series > upper)).sum())
    }

# 构建 TenureGroup：用户使用时长分层
tenure_bins = [0, 6, 12, 24, 36, 100]
tenure_labels = ["0-6月", "6-12月", "12-24月", "24-36月", "36月以上"]
cleaned_df["TenureGroup"] = pd.cut(
    cleaned_df["Tenure"], bins=tenure_bins, labels=tenure_labels,
    right=False, include_lowest=True
)

# 新建 IsMobileLogin：移动端为 1，其他设备为 0
cleaned_df["IsMobileLogin"] = (
    cleaned_df["PreferredLoginDevice"] == "Mobile Phone"
).astype(int)

# 生成 outlier_report（每行对应一个待检查字段）
outlier_fields = ["WarehouseToHome", "OrderCount", "CashbackAmount"]
outlier_report = pd.DataFrame([
    iqr_outlier_summary(cleaned_df[col]) for col in outlier_fields
], index=outlier_fields)
display(outlier_report)

,Q1,Q3,下限,上限,候选异常值数量
WarehouseToHome,9.00,20.00,-7.50,36.50,2
OrderCount,1.00,3.00,-2.00,6.00,703
CashbackAmount,145.77,196.39,69.84,272.33,438


### 任务 4：业务规则检查

统计以下不合规记录数，并写出你的处理结论：

- 使用时长小于 0；
- 仓库距离小于 0；
- 订单数小于或等于 0；
- 返现金额小于 0。

如果结果为 0，也应在项目日志或总结中记录。

In [9]:
# 业务规则检查
business_rule_report = pd.DataFrame({
    "规则": [
        "使用时长(Tenure) < 0",
        "仓库距离(WarehouseToHome) < 0",
        "订单数(OrderCount) <= 0",
        "返现金额(CashbackAmount) < 0",
    ],
    "不合规记录数": [
        int((cleaned_df["Tenure"] < 0).sum()),
        int((cleaned_df["WarehouseToHome"] < 0).sum()),
        int((cleaned_df["OrderCount"] <= 0).sum()),
        int((cleaned_df["CashbackAmount"] < 0).sum()),
    ]
})
display(business_rule_report)
# ============================================================
# 处理结论：
# ============================================================
# 1. Tenure < 0（使用时长为负）
#    → 物理上不可能，属于数据录入错误。若存在此类记录，应标记为异常、
#       通知业务方核实后修正或剔除；若为 0 条，也记录"已确认无异常"。
#
# 2. WarehouseToHome < 0（仓库距离为负）
#    → 距离不能为负，同上属于数据错误。若存在，建议追溯到原始数据源核查；
#       若为 0 条，记录"已确认无异常"。
#
# 3. OrderCount <= 0（订单数 ≤ 0）
#    → 需要分情况讨论：
#      - OrderCount = 0：可能是刚注册但从未下单的用户，业务上合理，
#        应保留，但可在分析时单独标注"无订单用户"分组；
#      - OrderCount < 0：数据错误，同上述处理。
#      因此不能一刀切删除 OrderCount=0 的记录，只在 <0 时才视为不合规。
#
# 4. CashbackAmount < 0（返现金额为负）
#    → 返现金额通常 ≥ 0（要么没返现，要么有返现），负值无业务含义。
#       若存在，视为数据错误，需业务方确认。
#
# 总体原则（贯彻项目规则）：
# - 先记录、再判断、不盲目删除 → 所有不合规记录先记入日志，
#   数值级错误（<0）可标记后剔除，业务级边界值（如 OrderCount=0）
#   保留并标注，供第五天分析时分层处理。

,规则,不合规记录数
0,使用时长(Tenure) < 0,0
1,仓库距离(WarehouseToHome) < 0,0
2,订单数(OrderCount) <= 0,0
3,返现金额(CashbackAmount) < 0,0


---
## 6. 项目验收与交付

请生成清洗后质量报告，比较清洗前后缺失值，并导出全部交付物。

In [10]:
# 最终验收
quality_after = build_quality_report(cleaned_df)

# 断言验证
assert cleaned_df[NUMERIC_MISSING_COLS].isna().sum().sum() == 0, \
    "数值字段仍有缺失值！"
assert "Phone" not in cleaned_df["PreferredLoginDevice"].unique(), \
    "PreferredLoginDevice 中仍存在 'Phone'！"
assert "COD" not in cleaned_df["PreferredPaymentMode"].unique(), \
    "PreferredPaymentMode 中仍存在 'COD'！"
assert "CC" not in cleaned_df["PreferredPaymentMode"].unique(), \
    "PreferredPaymentMode 中仍存在 'CC'！"
assert {"TenureGroup", "IsMobileLogin"}.issubset(cleaned_df.columns), \
    "缺少 TenureGroup 或 IsMobileLogin 列！"

print("所有断言通过！")

# 导出交付文件
quality_before.to_csv(OUTPUT_DIR / "data_quality_before.csv", index=True, encoding="utf-8-sig")
quality_after.to_csv(OUTPUT_DIR / "data_quality_after.csv", index=True, encoding="utf-8-sig")
cleaning_log.to_csv(OUTPUT_DIR / "cleaning_log.csv", index=False, encoding="utf-8-sig")
cleaned_df.to_csv(OUTPUT_DIR / "ecommerce_customer_cleaned.csv", index=False, encoding="utf-8-sig")

# 输出异常值报告和业务规则检查报告
print("\n=== 候选异常值报告 ===")
display(outlier_report)

print("\n=== 业务规则检查报告 ===")
display(business_rule_report)

# 输出交付文件路径
print("\n=== 交付文件 ===")
for f in OUTPUT_DIR.glob("*"):
    print(f"  {f}")

print("\n清洗完成！所有交付物已生成。")

所有断言通过！

=== 候选异常值报告 ===


,Q1,Q3,下限,上限,候选异常值数量
WarehouseToHome,9.00,20.00,-7.50,36.50,2
OrderCount,1.00,3.00,-2.00,6.00,703
CashbackAmount,145.77,196.39,69.84,272.33,438



=== 业务规则检查报告 ===


,规则,不合规记录数
0,使用时长(Tenure) < 0,0
1,仓库距离(WarehouseToHome) < 0,0
2,订单数(OrderCount) <= 0,0
3,返现金额(CashbackAmount) < 0,0



=== 交付文件 ===
  e:\pythonProject\day4\output\day04_project\cleaning_log.csv
  e:\pythonProject\day4\output\day04_project\data_quality_after.csv
  e:\pythonProject\day4\output\day04_project\data_quality_before.csv
  e:\pythonProject\day4\output\day04_project\ecommerce_customer_cleaned.csv

清洗完成！所有交付物已生成。


## 项目复盘

请在提交前用不超过 200 字回答：

1. 本项目发现了哪些数据质量问题？
2. 你对缺失值、类别不一致、候选异常值分别采取了什么策略？
3. 为什么清洗后的数据可以作为第五天分析的输入？
4. 哪些处理规则仍需要业务人员确认？

In [12]:
import sys, tensorflow as tf
print("当前使用的Python路径：", sys.executable)
print("TensorFlow版本：", tf.__version__)

当前使用的Python路径： e:\anaconda3\envs\tensorflow\python.exe
TensorFlow版本： 2.20.0
